# Enterprise Finance Data Generator

This notebook creates a synthetic enterprise finance dataset for a portfolio project using **Python, Pandas, NumPy, Faker, and Power BI**.

The workflow generates reusable dimension tables, a finance fact table, and a flattened `Master_Finance.csv` file for analysis in SQL Server and Power BI.

### Output
- `Dim_Date.csv`
- `Dim_Product.csv`
- `Dim_Customer.csv`
- `Dim_Salesperson.csv`
- `Dim_Region.csv`
- `Dim_Currency.csv`
- `Fact_Finance.csv`
- `Master_Finance.csv`

> All generated data is synthetic and intended only for learning and portfolio demonstration.

## 1. Setup

Import the required libraries, set reproducible random seeds, and create an output folder.

In [ ]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
from faker import Faker

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
Faker.seed(SEED)

fake = Faker()

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Environment ready")

## 2. Date Dimension

Create a daily date dimension covering 2022 through 2026. `DateKey` is stored in `YYYYMMDD` integer format for easy use in SQL Server and Power BI.

In [ ]:
dates = pd.date_range(
    start="2022-01-01",
    end="2026-12-31",
    freq="D"
)

dim_date = pd.DataFrame({
    "DateKey": dates.strftime("%Y%m%d").astype(int),
    "FullDate": dates,
    "DayNumber": dates.day,
    "DayName": dates.day_name(),
    "WeekNumber": dates.isocalendar().week.astype(int),
    "MonthNumber": dates.month,
    "MonthName": dates.month_name(),
    "QuarterNumber": dates.quarter,
    "YearNumber": dates.year,
    "IsWeekend": (dates.dayofweek >= 5).astype(int),
})

dim_date.head()

## 3. Product Dimension

Generate 500 synthetic products across seven engineering-oriented product categories.

In [ ]:
categories = [
    "Aftertreatment",
    "Sensors",
    "Turbochargers",
    "Fuel Systems",
    "Filtration",
    "Electronics",
    "Cooling Systems",
]

products = []

for product_key in range(1, 501):
    category = random.choice(categories)
    standard_cost = random.randint(50, 500)
    list_price = random.randint(max(standard_cost + 50, 100), 1000)

    products.append({
        "ProductKey": product_key,
        "ProductCode": f"P{product_key:04}",
        "ProductName": f"{category} Product {product_key}",
        "Category": category,
        "StandardCost": standard_cost,
        "ListPrice": list_price,
    })

dim_product = pd.DataFrame(products)
dim_product.head()

## 4. Customer Dimension

Generate 5,000 synthetic customers across European markets with customer type, industry, geography, and customer-since attributes.

In [ ]:
countries = [
    "Germany",
    "France",
    "Netherlands",
    "Belgium",
    "Italy",
    "Spain",
    "Sweden",
    "Poland",
    "Austria",
]

customers = []

for customer_key in range(1, 5001):
    customers.append({
        "CustomerKey": customer_key,
        "CustomerCode": f"C{customer_key:05}",
        "CustomerName": fake.company(),
        "CustomerType": random.choice(["OEM", "Distributor", "Fleet"]),
        "Industry": random.choice(["Automotive", "Manufacturing", "Transportation"]),
        "City": fake.city(),
        "Country": random.choice(countries),
        "CustomerSince": fake.date_between(start_date="-10y", end_date="today"),
        "IsActive": 1,
    })

dim_customer = pd.DataFrame(customers)
dim_customer.head()

## 5. Salesperson, Region, and Currency Dimensions

In [ ]:
salespeople = []

for salesperson_key in range(1, 121):
    salespeople.append({
        "SalespersonKey": salesperson_key,
        "EmployeeCode": f"E{salesperson_key:04}",
        "FirstName": fake.first_name(),
        "LastName": fake.last_name(),
        "DepartmentName": random.choice(["Sales", "Key Accounts", "Business Development"]),
        "Country": random.choice(countries),
        "HireDate": fake.date_between(start_date="-8y", end_date="today"),
        "IsActive": 1,
    })

dim_salesperson = pd.DataFrame(salespeople)

regions = [
    [1, "Germany", "Bavaria", "Munich", "EUR"],
    [2, "Germany", "Hesse", "Frankfurt", "EUR"],
    [3, "France", "Ile-de-France", "Paris", "EUR"],
    [4, "Netherlands", "North Holland", "Amsterdam", "EUR"],
    [5, "Belgium", "Brussels", "Brussels", "EUR"],
    [6, "Italy", "Lombardy", "Milan", "EUR"],
    [7, "Spain", "Madrid", "Madrid", "EUR"],
    [8, "Sweden", "Stockholm", "Stockholm", "SEK"],
    [9, "Poland", "Mazovia", "Warsaw", "PLN"],
    [10, "Austria", "Vienna", "Vienna", "EUR"],
]

dim_region = pd.DataFrame(
    regions,
    columns=["RegionKey", "Country", "RegionName", "City", "CurrencyCode"]
)

currencies = [
    [1, "EUR", "Euro", 1.00],
    [2, "SEK", "Swedish Krona", 0.088],
    [3, "PLN", "Polish Zloty", 0.23],
    [4, "GBP", "British Pound", 1.17],
    [5, "CHF", "Swiss Franc", 1.04],
]

dim_currency = pd.DataFrame(
    currencies,
    columns=["CurrencyKey", "CurrencyCode", "CurrencyName", "ExchangeRateToEUR"]
)

display(dim_salesperson.head())
display(dim_region)
display(dim_currency)

## 6. Finance Fact Table

Generate 250,000 synthetic finance transactions.

A valid `DateKey` is sampled directly from the date dimension rather than generated as a random integer. This guarantees that every transaction maps to a real calendar date.

In [ ]:
N_TRANSACTIONS = 250_000

date_keys = dim_date["DateKey"].to_numpy()

fact_finance = pd.DataFrame({
    "DateKey": np.random.choice(date_keys, size=N_TRANSACTIONS),
    "ProductKey": np.random.randint(1, 501, size=N_TRANSACTIONS),
    "CustomerKey": np.random.randint(1, 5001, size=N_TRANSACTIONS),
    "RegionKey": np.random.randint(1, 11, size=N_TRANSACTIONS),
    "SalespersonKey": np.random.randint(1, 121, size=N_TRANSACTIONS),
    "DepartmentKey": np.random.randint(1, 11, size=N_TRANSACTIONS),
    "CurrencyKey": np.random.randint(1, 6, size=N_TRANSACTIONS),
    "Quantity": np.random.randint(1, 21, size=N_TRANSACTIONS),
})

fact_finance["Revenue"] = np.round(
    np.random.uniform(100, 5000, size=N_TRANSACTIONS), 2
)

fact_finance["Cost"] = np.round(
    fact_finance["Revenue"] * np.random.uniform(0.60, 0.90, size=N_TRANSACTIONS), 2
)

fact_finance["Profit"] = np.round(
    fact_finance["Revenue"] - fact_finance["Cost"], 2
)

fact_finance["Budget"] = np.round(
    fact_finance["Revenue"] * np.random.uniform(0.95, 1.05, size=N_TRANSACTIONS), 2
)

fact_finance["Forecast"] = np.round(
    fact_finance["Revenue"] * np.random.uniform(0.95, 1.10, size=N_TRANSACTIONS), 2
)

fact_finance.head()

## 7. Create the Master Analytical Dataset

Merge product and customer attributes into the finance fact table to create the flattened dataset used in Power BI.

In [ ]:
master_finance = (
    fact_finance
    .merge(dim_product, on="ProductKey", how="left")
    .merge(dim_customer, on="CustomerKey", how="left")
)

master_finance.head()

## 8. Data Validation

Perform simple quality checks before exporting the files.

In [ ]:
print("Master rows:", len(master_finance))
print("Master columns:", master_finance.shape[1])
print("Duplicate rows:", master_finance.duplicated().sum())
print("Total missing values:", master_finance.isna().sum().sum())

financial_summary = pd.DataFrame({
    "Metric": ["Revenue", "Cost", "Profit", "Budget", "Forecast"],
    "Total": [
        master_finance["Revenue"].sum(),
        master_finance["Cost"].sum(),
        master_finance["Profit"].sum(),
        master_finance["Budget"].sum(),
        master_finance["Forecast"].sum(),
    ]
})

display(financial_summary)

## 9. Export CSV Files

Export the dimensional tables, finance fact table, and flattened master dataset.

In [ ]:
exports = {
    "Dim_Date.csv": dim_date,
    "Dim_Product.csv": dim_product,
    "Dim_Customer.csv": dim_customer,
    "Dim_Salesperson.csv": dim_salesperson,
    "Dim_Region.csv": dim_region,
    "Dim_Currency.csv": dim_currency,
    "Fact_Finance.csv": fact_finance,
    "Master_Finance.csv": master_finance,
}

for filename, dataframe in exports.items():
    dataframe.to_csv(OUTPUT_DIR / filename, index=False)

print(f"Exported {len(exports)} CSV files to: {OUTPUT_DIR.resolve()}")

## 10. Final Verification

In [ ]:
for filename in exports:
    path = OUTPUT_DIR / filename
    print(f"{filename:24} {path.stat().st_size / 1_000_000:8.2f} MB")

print("\nPortfolio dataset generation complete.")